In [4]:
!pip install -q langgraph langchain langchain-groq langchain-community pydantic python-dotenv tavily-python
print("✅ Packages installed")

✅ Packages installed



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from dotenv import load_dotenv
import os

load_dotenv()

groq_key = os.getenv("GROQ_API_KEY")
tavily_key = os.getenv("TAVILY_API_KEY")

print("GROQ_API_KEY found:", bool(groq_key))
print("TAVILY_API_KEY found:", bool(tavily_key))

assert groq_key, "Missing GROQ_API_KEY — check your .env file"
assert tavily_key, "Missing TAVILY_API_KEY — check your .env file"
print("✅ Keys loaded")

GROQ_API_KEY found: True
TAVILY_API_KEY found: True
✅ Keys loaded


In [8]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

response = llm.invoke("Reply with exactly: API connection successful.")
print(response.content)

API connection successful.


In [7]:
import requests

headers = {"Authorization": f"Bearer {groq_key}"}
resp = requests.get("https://api.groq.com/openai/v1/models", headers=headers)
models = resp.json()

for m in models["data"]:
    print(m["id"])

meta-llama/llama-prompt-guard-2-22m
whisper-large-v3-turbo
allam-2-7b
groq/compound-mini
openai/gpt-oss-120b
openai/gpt-oss-20b
whisper-large-v3
groq/compound
canopylabs/orpheus-v1-english
qwen/qwen3.8-27b
canopylabs/orpheus-arabic-saudi
openai/gpt-oss-safeguard-20b
meta-llama/llama-prompt-guard-2-86m


In [9]:
from tavily import TavilyClient

tavily = TavilyClient(api_key=tavily_key)

results = tavily.search(query="latest developments in AI agents 2026", max_results=3)

for r in results["results"]:
    print("-", r["title"], "|", r["url"])

print("✅ Search API working")

- Latest AI developments News | June, 2026 (STARTUP EDITION) | https://blog.mean.ceo/latest-ai-developments-news-june-2026
- Latest AI Developments: August 2026 Update - Local AI Zone | https://local-ai-zone.github.io/blog/ai-updates-august-2026.html
- What's Next in AI: Five Trends to Watch in 2026 | https://blog.bytebytego.com/p/whats-next-in-ai-five-trends-to-watch
✅ Search API working


In [10]:
from typing import TypedDict, Annotated, List, Dict, Optional
from operator import add

class ResearchState(TypedDict):
    query: str
    plan: List[str]
    search_results: Annotated[List[Dict], add]
    extracted_facts: Annotated[List[Dict], add]
    fact_check_results: Annotated[List[Dict], add]
    summaries: Annotated[List[Dict], add]
    final_report: Optional[str]
    agent_log: Annotated[List[str], add]

print("✅ ResearchState schema defined")
print(ResearchState.__annotations__)

✅ ResearchState schema defined
{'query': <class 'str'>, 'plan': typing.List[str], 'search_results': typing.Annotated[typing.List[typing.Dict], <built-in function add>], 'extracted_facts': typing.Annotated[typing.List[typing.Dict], <built-in function add>], 'fact_check_results': typing.Annotated[typing.List[typing.Dict], <built-in function add>], 'summaries': typing.Annotated[typing.List[typing.Dict], <built-in function add>], 'final_report': typing.Optional[str], 'agent_log': typing.Annotated[typing.List[str], <built-in function add>]}


In [11]:
from langgraph.graph import StateGraph, START, END

def planner_node(state: ResearchState) -> dict:
    print("Planner node running (calling Groq)...")
    prompt = f"Break this research question into exactly 1 concise sub-question: {state['query']}"
    response = llm.invoke(prompt)
    return {
        "plan": [response.content.strip()],
        "agent_log": ["planner_node: generated 1 sub-question via Groq"]
    }

def researcher_stub(state: ResearchState) -> dict:
    print("Researcher node running...")
    return {
        "search_results": [{"title": "dummy result", "url": "https://example.com"}],
        "agent_log": ["researcher_stub: found 1 dummy result"]
    }

graph = StateGraph(ResearchState)
graph.add_node("planner", planner_node)
graph.add_node("researcher", researcher_stub)

graph.add_edge(START, "planner")
graph.add_edge("planner", "researcher")
graph.add_edge("researcher", END)

app = graph.compile()
print(" Graph compiled")

 Graph compiled


In [12]:
result = app.invoke({
    "query": "Compare EV battery strategies of Tesla, BYD, and CATL",
    "plan": [],
    "search_results": [],
    "extracted_facts": [],
    "fact_check_results": [],
    "summaries": [],
    "final_report": None,
    "agent_log": []
})

print("\n--- Final State ---")
for key, value in result.items():
    print(f"{key}: {value}")

Planner node running (calling Groq)...
Researcher node running...

--- Final State ---
query: Compare EV battery strategies of Tesla, BYD, and CATL
plan: ['How do Tesla, BYD, and CATL differ in their EV battery strategies?']
search_results: [{'title': 'dummy result', 'url': 'https://example.com'}]
extracted_facts: []
fact_check_results: []
summaries: []
final_report: None
agent_log: ['planner_node: generated 1 sub-question via Groq', 'researcher_stub: found 1 dummy result']


In [ ]:
import os

folders = ["agents", "tools", "graph", "state", "tests", "data", "docs"]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    gitkeep = os.path.join(folder, ".gitkeep")
    if not os.path.exists(gitkeep):
        open(gitkeep, "w").close()

with open(".gitignore", "w") as f:
    f.write(".env\n__pycache__/\n*.pyc\nvenv/\n.ipynb_checkpoints/\n")

print("✅ Project structure created:")
for folder in folders:
    print(" -", folder + "/")
print(" - .gitignore")